# Tarea 3: aplicaciones con puntos de interés y descriptores (Cuaderno base)

**Procesamiento de Imágenes — Universidad Diego Portales**

Este cuaderno contiene dos aplicaciones web hechas con Gradio, que se usan desde el teléfono:

1. **Antes y después con detección de cambios:** alinea dos fotos de una misma escena y marca qué objetos
   aparecieron o desaparecieron.
2. **Panorama:** une 2 o 3 fotos tomadas girando el teléfono.

Las funciones marcadas como **Entregado** no se modifican. Las funciones que deben implementar aparecen con
su firma y `raise NotImplementedError`; sus nombres y argumentos deben quedar exactamente como están.

En Google Colab conviene activar la GPU (*Entorno de ejecución → Cambiar tipo de entorno*).

## Equipo y reparto

| Persona | Nombre | Parte |
|---|---|---|
| **Persona 1 — Núcleo** | *(nombre)* | Pasos 1, 2 y 3 · Tabla 1 |
| **Persona 2 — Cambios** | *(nombre)* | Pasos 4 y 5 · Tabla 2 · par de cambios propio |
| **Persona 3 — Panorama** | *(nombre)* | Paso 6 · Tabla 3 · panorama propio · Colab y video |

> Cada uno edita **solo las celdas de su parte**. Las celdas marcadas *Entregado* no se tocan. Detalle completo en el `README.md` del repo.

## 0. Preparación (Entregado)

In [ ]:
import sys

EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    !pip install -q gradio kornia

In [ ]:
import os
import time

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import kornia as K
import kornia.feature as KF
import gradio as gr
from PIL import Image, ImageOps

DISPOSITIVO = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo:", DISPOSITIVO)

# ---- Parámetros fijos: no los cambien, para que las mediciones sean comparables ----
LADO_MAX = 1024          # lado mayor de toda imagen, en píxeles
RAZON = 0.8              # prueba de razón de Lowe (SIFT y ORB)
UMBRAL_RANSAC = 5.0      # píxeles
MIN_INLIERS = 15         # bajo esto, la homografía no es confiable
METODOS = ["SIFT", "ORB", "DISK + LightGlue"]   # el tercero puede ser otro método aprendido

CARPETA = "imagenes"     # set común: imagenes/cambios y imagenes/panorama

### 0.1 Utilidades (Entregado)

| Función | Qué hace |
|---|---|
| `cargar(ruta_o_arreglo)` | Lee una imagen (o recibe un arreglo), corrige la orientación del teléfono, la pasa a RGB `uint8` y la reduce a `LADO_MAX` |
| `medir_tiempo(fn, *args)` | Ejecuta `fn(*args)` y devuelve `(resultado, milisegundos)` |
| `dibujar_pares(a, b, pa, pb, inliers)` | Las dos imágenes lado a lado con los pares que respetan la homografía |
| `cortina(antes, despues, posicion)` y `tablero(antes, despues)` | Combinaciones de dos imágenes del mismo tamaño |
| `mostrar(imagenes, titulos)` | Muestra una fila de imágenes |

In [ ]:
def cargar(entrada, lado=LADO_MAX):
    if isinstance(entrada, np.ndarray):
        im = entrada[..., :3]
    else:
        im = np.array(ImageOps.exif_transpose(Image.open(entrada)).convert("RGB"))
    e = min(1.0, lado / max(im.shape[:2]))
    if e < 1:
        im = cv2.resize(im, None, fx=e, fy=e, interpolation=cv2.INTER_AREA)
    return np.ascontiguousarray(im, dtype=np.uint8)


def medir_tiempo(fn, *args):
    if DISPOSITIVO.type == "cuda":
        torch.cuda.synchronize()
    inicio = time.perf_counter()
    resultado = fn(*args)
    if DISPOSITIVO.type == "cuda":
        torch.cuda.synchronize()
    return resultado, 1000 * (time.perf_counter() - inicio)


def dibujar_pares(a, b, pa, pb, inliers, maximo=80, alto=480):
    ea, eb = alto / a.shape[0], alto / b.shape[0]
    A, B = cv2.resize(a, None, fx=ea, fy=ea), cv2.resize(b, None, fx=eb, fy=eb)
    lienzo = np.full((alto, A.shape[1] + 10 + B.shape[1], 3), 255, np.uint8)
    lienzo[:, :A.shape[1]] = A
    lienzo[:, A.shape[1] + 10:] = B
    idx = np.nonzero(inliers)[0]
    if len(idx):
        idx = idx[np.linspace(0, len(idx) - 1, min(maximo, len(idx))).astype(int)]
    for i in idx:
        p = (int(pa[i, 0] * ea), int(pa[i, 1] * ea))
        q = (int(pb[i, 0] * eb) + A.shape[1] + 10, int(pb[i, 1] * eb))
        cv2.line(lienzo, p, q, (99, 210, 151), 1, cv2.LINE_AA)
        cv2.circle(lienzo, p, 3, (99, 210, 151), -1)
        cv2.circle(lienzo, q, 3, (99, 210, 151), -1)
    return lienzo


def cortina(antes, despues, posicion=50):
    x = int(despues.shape[1] * posicion / 100)
    salida = despues.copy()
    salida[:, :x] = antes[:, :x]
    salida[:, max(x - 2, 0):x + 2] = 255
    return salida


def tablero(antes, despues, lado=64):
    yy, xx = np.mgrid[:despues.shape[0], :despues.shape[1]]
    m = (yy // lado + xx // lado) % 2 == 0
    salida = despues.copy()
    salida[m] = antes[m]
    return salida


def mostrar(imagenes, titulos, alto=4.5):
    fig, ax = plt.subplots(1, len(imagenes), figsize=(alto * 1.5 * len(imagenes), alto), squeeze=False)
    for a, im, t in zip(ax[0], imagenes, titulos):
        a.imshow(im, cmap="gray")
        a.set_title(t)
        a.axis("off")
    fig.tight_layout()
    plt.show()

---
## Parte 1. Núcleo común

### Paso 1. `emparejar(a, b, metodo)`

Recibe dos imágenes RGB y el nombre de un método de `METODOS`. Devuelve `(pa, pb, n_a, n_b)`:

- `pa`, `pb`: arreglos `float32` de forma `(N, 2)` con las coordenadas $(x, y)$ de cada par, en píxeles de
  `a` y de `b`. La fila `i` de `pa` corresponde a la fila `i` de `pb`.
- `n_a`, `n_b`: cuántos puntos se detectaron en cada imagen.

Para SIFT y ORB, emparejen con el vecino más cercano y la prueba de razón (`RAZON`). Para el método aprendido
pueden usar DISK + LightGlue de `kornia`, u otro (LoFTR, SuperPoint + LightGlue). Los modelos se cargan **una
sola vez**, fuera de la función, para no medir su carga como tiempo.

> 👤 **Persona 1.** Subir primero **SIFT** (y los Pasos 2 y 3) para que las Personas 2 y 3 puedan probar; después ORB y el método aprendido.
> SIFT/ORB: `detectAndCompute` → `cv2.BFMatcher` (`NORM_L2` para SIFT, `NORM_HAMMING` para ORB) → `knnMatch(..., k=2)` → prueba de razón con `RAZON`.
> Aprendido: DISK + LightGlue de kornia (`KF.DISK`, `KF.LightGlueMatcher`); modelos en la celda de abajo, **no** dentro de `emparejar`.

In [ ]:
def emparejar(a, b, metodo):
    """Devuelve pa, pb (float32, N x 2), n_a y n_b."""
    raise NotImplementedError

In [ ]:
# Carguen aquí los modelos de su método aprendido (una sola vez)

### Paso 2. `estimar_homografia(pa, pb)`

Estima la homografía que lleva los puntos de `a` a los de `b` con
`cv2.findHomography(pa, pb, cv2.USAC_MAGSAC, UMBRAL_RANSAC)`. Devuelve `(H, inliers)`: `H` es la matriz de
3 × 3 (o `None` si no se pudo estimar) e `inliers` es un arreglo booleano de largo `N`. Con menos de 4 pares,
devuelve `None` y un arreglo de `False`.

> 👤 **Persona 1.**

In [ ]:
def estimar_homografia(pa, pb):
    """Devuelve H (3 x 3 o None) e inliers (bool, largo N)."""
    raise NotImplementedError

### Paso 3. `medidas(pa, pb, H, inliers, n_a, n_b, ms)`

Devuelve un diccionario con exactamente estas claves:

| Clave | Valor |
|---|---|
| `"puntos A"`, `"puntos B"` | `n_a`, `n_b` |
| `"pares"` | número de pares |
| `"inliers"` | número de inliers |
| `"% inliers"` | `100 · inliers / pares` (0 si no hay pares) |
| `"error (px)"` | error de reproyección de los inliers: $\sqrt{\frac{1}{n}\sum_i \lVert H(p_a^i) - p_b^i \rVert^2}$, o `NaN` si `H` es `None` |
| `"tiempo (ms)"` | `ms` |

Para aplicar `H` a los puntos usen `cv2.perspectiveTransform`.

> 👤 **Persona 1.** Ojo: el error es solo sobre los **inliers**.

In [ ]:
def medidas(pa, pb, H, inliers, n_a, n_b, ms):
    """Diccionario con las medidas de un par."""
    raise NotImplementedError

### `analizar_par(a, b, metodo)` (Entregado)

Une los tres pasos: empareja midiendo el tiempo, estima la homografía y calcula las medidas. Devuelve
`(H, pa, pb, inliers, fila)`, donde `fila` es el diccionario de `medidas` con el método agregado.

In [ ]:
def analizar_par(a, b, metodo):
    (pa, pb, n_a, n_b), ms = medir_tiempo(emparejar, a, b, metodo)
    H, inliers = estimar_homografia(pa, pb)
    fila = {"método": metodo, **medidas(pa, pb, H, inliers, n_a, n_b, ms)}
    return H, pa, pb, inliers, fila

---
## Parte 2. Antes y después con detección de cambios

### Paso 4. `alinear(antes, despues, H)`

Lleva `antes` a la perspectiva de `despues` con `cv2.warpPerspective`. Devuelve `(antes_alineada, valida)`:
`antes_alineada` tiene el mismo tamaño que `despues`, y `valida` es una máscara booleana con los píxeles que
quedan cubiertos por la imagen deformada (deformen también una imagen de unos).

> 👤 **Persona 2.**

In [ ]:
def alinear(antes, despues, H):
    """Devuelve antes_alineada (RGB) y valida (bool)."""
    raise NotImplementedError

### Paso 5. `detectar_cambios(antes_alineada, despues, valida, umbral, k_morf, area_min)`

Devuelve `(mascara, cajas)`: una máscara `uint8` en {0, 1} con los píxeles que cambiaron, y una lista de
rectángulos `(x, y, ancho, alto)`, uno por cambio. Pasos sugeridos:

1. Pasar ambas imágenes a gris y suavizar (por ejemplo, gaussiano de 5 × 5).
2. **Compensar la iluminación:** igualar la media y la desviación estándar de `antes_alineada` a las de `despues`,
   calculadas solo dentro de `valida`.
3. Diferencia absoluta y umbral `umbral`.
4. Limpiar con morfología: apertura y cierre con un elemento de `k_morf` × `k_morf`.
5. Descartar el borde de la zona válida (erosionar `valida` unos píxeles) y las componentes de área menor a
   `area_min`.

> 👤 **Persona 2.** Los tres parámetros se mueven desde la app (sliders): no fijarlos dentro de la función.

In [ ]:
def detectar_cambios(antes_alineada, despues, valida, umbral=40, k_morf=7, area_min=400):
    """Devuelve mascara (uint8 en {0, 1}) y cajas [(x, y, w, h), ...]."""
    raise NotImplementedError

### `dibujar_cambios` y `app_cambios` (Entregado)

`app_cambios` es la función que llama la interfaz: une los pasos y prepara todas las salidas.

In [ ]:
def dibujar_cambios(despues, mascara, cajas):
    salida = despues.copy()
    salida[mascara > 0] = (0.5 * salida[mascara > 0] + 0.5 * np.array([209, 67, 47])).astype(np.uint8)
    for x, y, w, h in cajas:
        cv2.rectangle(salida, (x, y), (x + w, y + h), (255, 230, 0), 3)
    return salida


def app_cambios(foto_antes, foto_despues, metodo, umbral, k_morf, area_min, posicion=50):
    if foto_antes is None or foto_despues is None:
        raise gr.Error("Falta una de las dos fotos.")
    a, b = cargar(foto_antes), cargar(foto_despues)
    H, pa, pb, inliers, fila = analizar_par(a, b, metodo)
    texto = (f"**{metodo}:** {fila['pares']} pares, **{fila['inliers']} inliers** "
             f"({fila['% inliers']:.0f} %), error {fila['error (px)']:.2f} px, {fila['tiempo (ms)']:.0f} ms.")
    pares = dibujar_pares(a, b, pa, pb, inliers)
    if H is None or fila["inliers"] < MIN_INLIERS:
        return pares, None, None, None, None, texto + f"\n\nMenos de {MIN_INLIERS} inliers: la alineación no es confiable.", None
    alineada, valida = alinear(a, b, H)
    mascara, cajas = detectar_cambios(alineada, b, valida, umbral, k_morf, area_min)
    texto += f"\n\n**Cambios detectados: {len(cajas)}**"
    return (pares, cortina(alineada, b, posicion), tablero(alineada, b),
            (0.5 * alineada + 0.5 * b).astype(np.uint8), dibujar_cambios(b, mascara, cajas), texto,
            (alineada, b))


def mover_cortina(par, posicion):
    # Mover la barra no vuelve a emparejar: usa el par ya alineado que guardó app_cambios
    if par is None:
        return None
    return cortina(par[0], par[1], posicion)

---
## Parte 3. Panorama

### Paso 6. `panorama(imagenes, metodo)`

Recibe una lista de 2 o 3 imágenes ordenadas de izquierda a derecha. Devuelve `(pano, filas)`: el panorama RGB
`uint8` y una lista con las medidas (`fila` de `analizar_par`) de cada par usado.

1. **Referencia:** la imagen central, `imagenes[len(imagenes) // 2]`.
2. **Homografías:** para cada otra imagen, estimen con `analizar_par` la homografía que la lleva a la referencia.
   Si alguna tiene menos de `MIN_INLIERS` inliers, lancen `ValueError` con un mensaje claro.
3. **Lienzo:** apliquen cada homografía a las 4 esquinas de su imagen (`cv2.perspectiveTransform`), calculen el
   rectángulo que las contiene a todas y una traslación $T$ que deje ese rectángulo en coordenadas positivas.
   Si el lienzo mide más de 4 · `LADO_MAX` por lado, lancen `ValueError`: la homografía es mala.
4. **Deformar y mezclar:** deformen cada imagen con $T \cdot H$ y promedien los píxeles donde se superponen.

> 👤 **Persona 3.** ⚠️ `app_panorama` lee `f['par']` de cada fila: agregar a cada `fila` la clave `"par"` (p. ej. `"1a2"`, `"3a2"`), además de las medidas.

In [ ]:
def panorama(imagenes, metodo):
    """Devuelve pano (RGB uint8) y la lista de medidas de cada par."""
    raise NotImplementedError

### `panorama_opencv` y `app_panorama` (Entregado)

`panorama_opencv` usa `cv2.Stitcher`, la solución completa de OpenCV, como referencia de comparación.

In [ ]:
def panorama_opencv(imagenes):
    estado, pano = cv2.Stitcher_create(cv2.Stitcher_PANORAMA).stitch([cv2.cvtColor(im, cv2.COLOR_RGB2BGR) for im in imagenes])
    if estado != cv2.Stitcher_OK:
        return None
    return cv2.cvtColor(pano, cv2.COLOR_BGR2RGB)


def app_panorama(foto_1, foto_2, foto_3, metodo):
    fotos = [cargar(f) for f in (foto_1, foto_2, foto_3) if f is not None]
    if len(fotos) < 2:
        raise gr.Error("Se necesitan al menos 2 fotos.")
    try:
        (pano, filas), ms = medir_tiempo(panorama, fotos, metodo)
    except ValueError as e:
        raise gr.Error(str(e))
    texto = "\n".join(f"- Par {f['par']}: {f['inliers']} inliers ({f['% inliers']:.0f} %), error {f['error (px)']:.2f} px"
                      for f in filas)
    texto = f"**{metodo}:** {pano.shape[1]} × {pano.shape[0]} px, {ms:.0f} ms en total.\n\n{texto}"
    return pano, panorama_opencv(fotos), texto

---
## Parte 4. Tablas comparativas

Los nombres de archivo del set común son:

- `imagenes/cambios/{facil,medio,dificil}_antes.jpeg` y `..._despues.jpeg`
- `imagenes/panorama/{1,2,3}.jpg`, de izquierda a derecha

Agreguen sus propias fotos en `imagenes/propias/` y súmenlas a las listas `PARES` y `PANORAMAS`.

> 👤 **Persona 2** sube su par a `imagenes/propias/` y descomenta `PARES["propio"]`. **Persona 3** sube su panorama y descomenta `PANORAMAS["propio"]` (ajustar nombres y extensión de archivo).
>
> ⚠️ El set común entregado **no trae el par `dificil`** (solo `facil` y `medio`). Conseguirlo con el profesor, o quitar `"dificil"` de la lista mientras tanto.

In [ ]:
PARES = {nivel: (f"{CARPETA}/cambios/{nivel}_antes.jpeg", f"{CARPETA}/cambios/{nivel}_despues.jpeg")
         for nivel in ("facil", "medio", "dificil")}
PANORAMAS = {"comun": [f"{CARPETA}/panorama/{i}.jpeg" for i in (1, 2, 3)]}
# PARES["propio"] = (f"{CARPETA}/propias/antes.jpeg", f"{CARPETA}/propias/despues.jpeg")
# PANORAMAS["propio"] = [f"{CARPETA}/propias/pano_{i}.jpg" for i in (1, 2, 3)]

### Paso 7. Tabla 1: comparación de métodos en los pares de cambios

Para cada par de `PARES` y cada método de `METODOS`, llamen a `analizar_par` y agreguen una columna `"alineó"`
(`"sí"` o `"no"`), decidida mirando la cortina o el tablero. Muestren la tabla como `DataFrame`.

> 👤 **Persona 1.** Indicar si los tiempos son en CPU o GPU (`DISPOSITIVO`).

In [ ]:
# Tabla 1

### Paso 8. Tabla 2: detección de cambios

Con el mejor método de la Tabla 1, apliquen `detectar_cambios` a cada par. Para cada par indiquen cuántos
objetos cambiaron realmente (contados por ustedes), cuántos detectó la aplicación, cuántos de esos son
correctos y cuántas detecciones son falsas. Muestren además la imagen con los cambios marcados.

> 👤 **Persona 2.** Usar el método que mejor alinea cada par según la Tabla 1.

In [ ]:
# Tabla 2

### Paso 9. Tabla 3: panorama

Para cada panorama de `PANORAMAS` y cada método, construyan el panorama y registren, por cada par, inliers,
`% inliers` y error, además del tiempo total, el tamaño del panorama y si funcionó (`"sí"` o `"no"`).
Agreguen una fila para `panorama_opencv` con su tiempo y tamaño. Muestren los panoramas obtenidos.

> 👤 **Persona 3.** Incluir la fila de `cv2.Stitcher` (`panorama_opencv`) y mostrar los panoramas.

In [ ]:
# Tabla 3

---
## Parte 5. La aplicación (Entregado)

En Colab, `share=True` entrega un enlace `https://....gradio.live` que se abre en el teléfono. El enlace funciona
mientras este cuaderno esté en ejecución.

In [ ]:
with gr.Blocks(title="Tarea 3") as demo:
    gr.Markdown("# Tarea 3: puntos de interés y descriptores")
    with gr.Tab("Antes y después"):
        with gr.Row():
            c_antes = gr.Image(label="Antes", type="numpy", sources=["upload", "webcam", "clipboard"])
            c_despues = gr.Image(label="Después", type="numpy", sources=["upload", "webcam", "clipboard"])
        c_metodo = gr.Radio(METODOS, value=METODOS[0], label="Método")
        with gr.Accordion("Parámetros de la detección de cambios", open=False):
            c_umbral = gr.Slider(5, 120, value=40, step=1, label="Umbral de diferencia")
            c_k = gr.Slider(1, 31, value=7, step=2, label="Tamaño del elemento morfológico")
            c_area = gr.Slider(0, 5000, value=400, step=50, label="Área mínima (px)")
        c_boton = gr.Button("Alinear y detectar cambios", variant="primary")
        c_texto = gr.Markdown()
        with gr.Tab("Cambios"):
            c_cambios = gr.Image(label="Cambios detectados")
        with gr.Tab("Cortina"):
            c_posicion = gr.Slider(0, 100, value=50, step=1, label="Posición de la cortina (%)")
            c_cortina = gr.Image(label="Izquierda: antes · derecha: después")
        with gr.Tab("Tablero"):
            c_tablero = gr.Image()
        with gr.Tab("Mezcla"):
            c_mezcla = gr.Image()
        with gr.Tab("Pares"):
            c_pares = gr.Image(label="Pares que respetan la homografía")
        c_par = gr.State()
        c_boton.click(app_cambios, [c_antes, c_despues, c_metodo, c_umbral, c_k, c_area, c_posicion],
                      [c_pares, c_cortina, c_tablero, c_mezcla, c_cambios, c_texto, c_par])
        c_posicion.change(mover_cortina, [c_par, c_posicion], c_cortina)
    with gr.Tab("Panorama"):
        with gr.Row():
            p_1 = gr.Image(label="Foto 1 (izquierda)", type="numpy", sources=["upload", "webcam"])
            p_2 = gr.Image(label="Foto 2", type="numpy", sources=["upload", "webcam"])
            p_3 = gr.Image(label="Foto 3 (derecha, opcional)", type="numpy", sources=["upload", "webcam"])
        p_metodo = gr.Radio(METODOS, value=METODOS[0], label="Método")
        p_boton = gr.Button("Construir panorama", variant="primary")
        p_texto = gr.Markdown()
        p_pano = gr.Image(label="Panorama")
        p_cv = gr.Image(label="Referencia: cv2.Stitcher")
        p_boton.click(app_panorama, [p_1, p_2, p_3, p_metodo], [p_pano, p_cv, p_texto])

demo.launch(share=EN_COLAB, debug=EN_COLAB, show_error=True)